# Fine-tune a CRNN + CTC model for Polish handwriting (HTR)

An open-source, **CTC-based** alternative to the TrOCR notebook. Where TrOCR is a
seq2seq transformer (a BART decoder bolted onto a ViT) that *hallucinates* plausible
Polish words when it has little data, a **CRNN+CTC** model reads the image column by
column and is far more **data-efficient** — the standard serious open-source path for
historical HTR (the same family as PyLaia and Kraken).

This is a self-contained PyTorch implementation, so there's no PyLaia/Kraken/eScriptorium
install or data-format wrangling to fight with in Colab. It trains on the **same**
`sample_1000` data you already uploaded for the TrOCR run.

**Before running:**
1. `Runtime > Change runtime type` and select a **GPU** (T4 is fine).
2. Make sure `sample_1000.zip` and `sample_1000_transcribed_v2.txt` are on your Drive
   (same files as the TrOCR notebook — see Step 1).

> **Note on data:** only ~980 of the 1000 lines have a non-empty human transcription.
> CRNN+CTC will do much better than TrOCR on this amount, but if you can label more
> lines from `output/3_lines/` it will keep improving — point `EXTRA_TRANSCRIPTIONS`
> at any extra `image\ttext` files in Step 3.

## Step 0 — Install dependencies

In [1]:
# Core deps are already on Colab (torch, torchvision, PIL, numpy). We add editdistance
# for fast CER/WER. The optional KenLM language-model decoder is installed later, in the
# clearly-marked optional section at the very end.
!pip install -q editdistance

## Step 1 — Get the data into Colab

Identical to the TrOCR notebook. Upload to your Drive (e.g. `MyDrive/UMwTI/`):

- `sample_1000.zip` — zip of the `output/sample_1000/` folder (1000 PNG line images)
- `sample_1000_transcribed_v2.txt` — tab-separated `image_name<TAB>text`

Set `DATA_DIR` to that Drive folder and run the cell.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, zipfile

# Drive folder where you uploaded the data
DATA_DIR = Path('/content/drive/MyDrive/UMwTI')

WORK_DIR = Path('/content/data')
WORK_DIR.mkdir(parents=True, exist_ok=True)

IMAGES_DIR = WORK_DIR / 'sample_1000'
TRANSCRIPTIONS_FILE = WORK_DIR / 'sample_1000_transcribed_v2.txt'

if not IMAGES_DIR.exists():
    zip_path = DATA_DIR / 'sample_1000.zip'
    print(f'Extracting {zip_path} ...')
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(WORK_DIR)
    if not IMAGES_DIR.exists():
        IMAGES_DIR = WORK_DIR  # zip had no sample_1000/ subfolder

if not TRANSCRIPTIONS_FILE.exists():
    shutil.copy(DATA_DIR / 'sample_1000_transcribed_v2.txt', TRANSCRIPTIONS_FILE)

n_images = len(list(IMAGES_DIR.glob('*.png')))
print('Images dir:', IMAGES_DIR, '-', n_images, 'images')
print('Transcriptions file:', TRANSCRIPTIONS_FILE)

Mounted at /content/drive
Extracting /content/drive/MyDrive/UMwTI/sample_1000.zip ...
Images dir: /content/data/sample_1000 - 1000 images
Transcriptions file: /content/data/sample_1000_transcribed_v2.txt


## Step 2 — Config & imports

In [3]:
from __future__ import annotations
import math, random, logging, time
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageOps, ImageFilter
import torchvision.transforms.functional as TF
import editdistance

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)-7s | %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger(__name__)

# ---- Image geometry ----
IMG_HEIGHT = 64        # all line crops are resized to this height
MAX_WIDTH  = 1024      # cap width (keeps batch memory sane); wider lines are squashed
DOWNSAMPLE = 4         # the CNN reduces width by this factor -> CTC time steps = W/4

# ---- Training ----
# NOTE on batch size: with only ~900 training lines, a big batch means very few
# optimizer steps per epoch (batch 64 -> ~14 steps/epoch), which starves a from-scratch
# CRNN of the updates it needs to escape the early 'predict only blank' CTC collapse.
# 32 is a good balance of speed and enough steps. Don't push this much higher unless you
# also raise EPOCHS a lot. (VRAM headroom is NOT the thing to optimise here; steps are.)
BATCH_SIZE    = 32
EPOCHS        = 150    # CRNN epochs are ~10s each; more epochs is cheap and helps a lot
LEARNING_RATE = 3e-4   # peak LR, reached after a short warmup (see Step 8)
WARMUP_EPOCHS = 5      # ramp LR 0 -> peak so the model doesn't dive into the blank solution
WEIGHT_DECAY  = 1e-4
VAL_SIZE      = 0.10
SEED          = 42
GRAD_CLIP     = 5.0
NUM_WORKERS   = 4      # Colab Pro gives more vCPUs; 4 is safe, try 8 if available
USE_AMP       = False  # fp16 + CTC is unstable and saves little on an 8M-param model
PRINT_TEST_EXAMPLES = 15

# Where the trained model is saved (on Drive, survives the session)
OUTPUT_MODEL_DIR = Path('/content/drive/MyDrive/UMwTI/models/crnn_ctc_sample1000')

# Optional: extra labeled files (image_name<TAB>text), same format as v2. Leave empty to use only sample_1000.
EXTRA_TRANSCRIPTIONS = []  # e.g. [Path('/content/data/more_labels.txt')]

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

Using device: cuda


## Step 3 — Load labeled samples & build the character vocabulary

Lines with an empty transcription are skipped. Unlike TrOCR (which used a fixed 50k
subword vocab), CTC learns a **character** alphabet built directly from your data —
index 0 is reserved for the CTC *blank*.

In [4]:
def load_samples(images_dir: Path, transcription_file: Path) -> list[dict]:
    records, skipped = [], 0
    with open(transcription_file, 'r', encoding='utf-8') as f:
        for idx, raw in enumerate(f, 1):
            line = raw.rstrip('\n')
            if not line.strip() or '\t' not in line:
                continue
            name, text = line.split('\t', 1)
            if not text.strip():
                skipped += 1; continue
            p = images_dir / name
            if not p.exists():
                # tolerate a missing image rather than aborting the whole run
                skipped += 1; continue
            records.append({'image_path': str(p), 'text': text})
    logger.info('Loaded %d samples from %s (%d skipped)', len(records), transcription_file.name, skipped)
    return records

records = load_samples(IMAGES_DIR, TRANSCRIPTIONS_FILE)
for extra in EXTRA_TRANSCRIPTIONS:
    records += load_samples(IMAGES_DIR, Path(extra))
assert records, 'No labeled samples loaded!'

# Build the alphabet from all transcriptions. idx 0 = CTC blank.
charset = sorted({c for r in records for c in r['text']})
BLANK = 0
char_to_idx = {c: i + 1 for i, c in enumerate(charset)}
idx_to_char = {i + 1: c for i, c in enumerate(charset)}
NUM_CLASSES = len(charset) + 1  # + blank
print(f'Alphabet size: {len(charset)} chars  ->  {NUM_CLASSES} classes (incl. blank)')
print('Chars:', ''.join(charset))

def encode_text(text: str) -> list[int]:
    return [char_to_idx[c] for c in text if c in char_to_idx]

Alphabet size: 109 chars  ->  110 classes (incl. blank)
Chars:  !"'(),-./0123456789:;=?ABCDEFGHIJKLMNOPRSTUVWXYZ\abcdefghijklmnopqrstuvwxyz|°½ßéóăĄąćęŁłńŚśźŻżбгепрстуъ–—”„…


## Step 4 — Train / validation split

In [5]:
rng = random.Random(SEED)
idxs = list(range(len(records)))
rng.shuffle(idxs)
n_val = max(1, int(len(records) * VAL_SIZE))
val_idx = set(idxs[:n_val])
train_records = [records[i] for i in idxs[n_val:]]
val_records   = [records[i] for i in idxs[:n_val]]
print(f'{len(train_records)} train / {len(val_records)} val')

882 train / 98 val


## Step 5 — Dataset, augmentation & batching

Each crop is converted to grayscale and resized to a fixed **height** (64px) while
keeping its aspect ratio, then right-padded with white so a batch shares one width.
The CTC *input length* for each sample is its real (unpadded) width // 4, so padding
never confuses the loss. Light, label-preserving augmentation (small rotation, scale,
perspective, ink erosion/dilation, brightness) is applied during training only — this
matters a lot with ~1k lines.

In [6]:
def resize_keep_height(img: Image.Image, height: int, max_width: int) -> Image.Image:
    w, h = img.size
    new_w = max(1, int(round(w * height / h)))
    new_w = min(new_w, max_width)
    return img.resize((new_w, height), Image.BILINEAR)

def augment(img: Image.Image) -> Image.Image:
    # small rotation
    if random.random() < 0.5:
        img = img.rotate(random.uniform(-2.5, 2.5), resample=Image.BILINEAR, fillcolor=255, expand=False)
    # random horizontal scale (stretch/squeeze)
    if random.random() < 0.4:
        w, h = img.size
        s = random.uniform(0.85, 1.15)
        img = img.resize((max(1, int(w * s)), h), Image.BILINEAR)
    # ink erosion / dilation
    if random.random() < 0.3:
        img = img.filter(ImageFilter.MaxFilter(3) if random.random() < 0.5 else ImageFilter.MinFilter(3))
    # brightness jitter
    if random.random() < 0.4:
        img = TF.adjust_brightness(img, random.uniform(0.8, 1.2))
    return img

class LineDataset(Dataset):
    def __init__(self, records, training: bool):
        self.records = records
        self.training = training
    def __len__(self):
        return len(self.records)
    def __getitem__(self, i):
        r = self.records[i]
        img = Image.open(r['image_path']).convert('L')  # grayscale
        img = resize_keep_height(img, IMG_HEIGHT, MAX_WIDTH)
        if self.training:
            img = augment(img)
            img = resize_keep_height(img, IMG_HEIGHT, MAX_WIDTH)  # augmentation may change H
        x = TF.to_tensor(img)              # (1, H, W) in [0,1]
        x = TF.normalize(x, [0.5], [0.5])  # -> [-1, 1]
        target = torch.tensor(encode_text(r['text']), dtype=torch.long)
        return x, target, x.shape[2], len(target), r['text']

def collate(batch):
    imgs, targets, widths, tlens, texts = zip(*batch)
    max_w = max(widths)
    padded = torch.full((len(imgs), 1, IMG_HEIGHT, max_w), 1.0)  # pad with white (=1 after norm? no)
    # white in [0,1] is 1.0 -> after normalize((0.5),(0.5)) white = (1-0.5)/0.5 = 1.0, ok
    for i, im in enumerate(imgs):
        padded[i, :, :, :im.shape[2]] = im
    input_lengths = torch.tensor([max(1, w // DOWNSAMPLE) for w in widths], dtype=torch.long)
    target_lengths = torch.tensor(tlens, dtype=torch.long)
    targets_concat = torch.cat(targets) if targets else torch.zeros(0, dtype=torch.long)
    return padded, targets_concat, input_lengths, target_lengths, list(texts)

train_ds = LineDataset(train_records, training=True)
val_ds   = LineDataset(val_records, training=False)
# pin_memory + persistent_workers + prefetch keep the GPU fed so it isn't idling between
# batches. persistent_workers avoids re-spawning workers every epoch (big win for many epochs).
_loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=(device == 'cuda'))
if NUM_WORKERS > 0:
    _loader_kw.update(persistent_workers=True, prefetch_factor=4)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, drop_last=False, **_loader_kw)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate, **_loader_kw)
print('Batches:', len(train_loader), 'train /', len(val_loader), 'val')

Batches: 28 train / 4 val


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## Step 6 — The CRNN model

A VGG-style CNN encoder collapses the 64px height to 1 while downsampling the width by
4, producing one feature vector per ~4 input columns. A 2-layer **bidirectional LSTM**
models left/right context, and a linear head emits per-timestep class logits for CTC.

In [7]:
class CRNN(nn.Module):
    def __init__(self, num_classes: int, img_height: int = 64, in_ch: int = 1, lstm_hidden: int = 256):
        super().__init__()
        def conv_bn(i, o, k=3, s=1, p=1, bn=True):
            layers = [nn.Conv2d(i, o, k, s, p)]
            if bn: layers.append(nn.BatchNorm2d(o))
            layers.append(nn.ReLU(inplace=True))
            return layers
        self.cnn = nn.Sequential(
            *conv_bn(in_ch, 64, bn=False), nn.MaxPool2d(2, 2),     # 64 -> 32 (H), W/2
            *conv_bn(64, 128, bn=False),   nn.MaxPool2d(2, 2),     # 32 -> 16 (H), W/4
            *conv_bn(128, 256), *conv_bn(256, 256), nn.MaxPool2d((2, 1), (2, 1)),  # 16 -> 8 (H)
            *conv_bn(256, 512), *conv_bn(512, 512), nn.MaxPool2d((2, 1), (2, 1)),  # 8 -> 4 (H)
            *conv_bn(512, 512), nn.MaxPool2d((2, 1), (2, 1)),      # 4 -> 2 (H)
            *conv_bn(512, 512), nn.MaxPool2d((2, 1), (2, 1)),      # 2 -> 1 (H)
        )
        self.rnn = nn.LSTM(512, lstm_hidden, num_layers=2, bidirectional=True, batch_first=False, dropout=0.2)
        self.fc = nn.Linear(lstm_hidden * 2, num_classes)
    def forward(self, x):
        f = self.cnn(x)                       # (B, 512, 1, T)
        assert f.size(2) == 1, f'height not collapsed to 1, got {f.size(2)}'
        f = f.squeeze(2)                      # (B, 512, T)
        f = f.permute(2, 0, 1)                # (T, B, 512)
        f, _ = self.rnn(f)                    # (T, B, 2*hidden)
        logits = self.fc(f)                   # (T, B, num_classes)
        return logits

model = CRNN(NUM_CLASSES, IMG_HEIGHT).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'CRNN parameters: {n_params/1e6:.1f}M')  # ~8M, vs ~330M for TrOCR-base

CRNN parameters: 12.4M


## Step 7 — CTC loss, greedy decoder, and CER/WER metrics

In [8]:
ctc_loss = nn.CTCLoss(blank=BLANK, zero_infinity=True)

def greedy_decode(logits: torch.Tensor, input_lengths: torch.Tensor) -> list[str]:
    # logits: (T, B, C). Argmax, collapse repeats, drop blanks.
    best = logits.argmax(2).permute(1, 0).cpu().numpy()  # (B, T)
    outs = []
    for b, seq in enumerate(best):
        L = int(input_lengths[b])
        prev, chars = BLANK, []
        for t in range(min(L, len(seq))):
            c = seq[t]
            if c != BLANK and c != prev:
                chars.append(idx_to_char.get(int(c), ''))
            prev = c
        outs.append(''.join(chars))
    return outs

def cer(refs, preds):
    d = sum(editdistance.eval(r, p) for r, p in zip(refs, preds))
    n = sum(max(1, len(r)) for r in refs)
    return d / n

def wer(refs, preds):
    d = n = 0
    for r, p in zip(refs, preds):
        rw, pw = r.split(), p.split()
        d += editdistance.eval(rw, pw); n += max(1, len(rw))
    return d / n

### Tuning VRAM usage / speed

Run the cell below during/after the first training epoch to see real GPU memory use.
If `nvidia-smi` shows the T4 well under ~12 GB, raise `BATCH_SIZE` (Step 2) and re-run
from Step 5 — bigger batches keep the GPU busy and cut wall-clock time per epoch.
Conversely, if you hit a CUDA out-of-memory error, lower `BATCH_SIZE` or `MAX_WIDTH`.

In [9]:
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv

memory.used [MiB], memory.total [MiB], utilization.gpu [%]
191 MiB, 15360 MiB, 0 %


## Step 8 — Train

AdamW + `ReduceLROnPlateau` on validation CER. CRNN is small (~8M params) so each
epoch is quick; the best checkpoint (lowest val CER) is kept in memory and saved in
Step 10.

In [10]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
# Schedule on validation LOSS, not CER: CER is stuck at 1.0 during the early blank-collapse
# phase, so scheduling on it would (wrongly) decay the LR to zero before the model escapes.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=8, min_lr=1e-5)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

steps_per_epoch = max(1, len(train_loader))
warmup_steps = WARMUP_EPOCHS * steps_per_epoch
global_step = 0

@torch.no_grad()
def evaluate(loader):
    model.eval()
    refs, preds = [], []
    total_loss, n = 0.0, 0
    for imgs, targets, in_len, tgt_len, texts in loader:
        imgs, t = imgs.to(device), targets.to(device)
        logits = model(imgs)
        # the CNN's real time dim may differ by +-1 from W//4; clamp input_lengths to it
        T = logits.size(0)
        in_len_c = torch.clamp(in_len, max=T)
        log_probs = F.log_softmax(logits, dim=2)
        loss = ctc_loss(log_probs, t, in_len_c.to(device), tgt_len.to(device))
        total_loss += loss.item() * imgs.size(0); n += imgs.size(0)
        preds += greedy_decode(logits, in_len_c)
        refs += list(texts)
    return total_loss / n, cer(refs, preds), wer(refs, preds), refs, preds

best_cer = float('inf')
best_state = None
history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0; t0 = time.time()
    for imgs, targets, in_len, tgt_len, texts in train_loader:
        # linear LR warmup for the first WARMUP_EPOCHS (prevents diving into the blank solution)
        if global_step < warmup_steps:
            warm_lr = LEARNING_RATE * (global_step + 1) / warmup_steps
            for g in optimizer.param_groups:
                g['lr'] = warm_lr
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(imgs)                       # (T, B, C)
            T = logits.size(0)
            in_len_c = torch.clamp(in_len, max=T).to(device)
            log_probs = F.log_softmax(logits, dim=2)
            loss = ctc_loss(log_probs, targets, in_len_c, tgt_len.to(device))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer); scaler.update()
        running += loss.item() * imgs.size(0)
        global_step += 1
    train_loss = running / len(train_ds)
    val_loss, val_cer, val_wer, _, val_preds = evaluate(val_loader)
    if global_step >= warmup_steps:
        scheduler.step(val_loss)        # only start decaying once warmup is done
    history.append((epoch, train_loss, val_loss, val_cer, val_wer))
    lr_now = optimizer.param_groups[0]['lr']
    # fraction of val lines where the model emitted *something* (not blank) -> escapes collapse
    nonblank = sum(1 for p in val_preds if p.strip()) / max(1, len(val_preds))
    flag = ''
    if val_cer < best_cer:
        best_cer = val_cer
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        flag = '  <- best'
    print(f'epoch {epoch:3d} | tr_loss {train_loss:6.3f} | val_loss {val_loss:6.3f} | CER {val_cer:.4f} | WER {val_wer:.4f} | non-blank {nonblank:4.0%} | lr {lr_now:.1e} | {time.time()-t0:4.0f}s{flag}')

if best_state is not None:
    model.load_state_dict(best_state)
print(f'\nBest validation CER: {best_cer:.4f}')

/tmp/ipykernel_1028/3146970291.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/tmp/ipykernel_1028/3146970291.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


epoch   1 | tr_loss 16.610 | val_loss 13.471 | CER 1.0000 | WER 1.0000 | non-blank   0% | lr 6.0e-05 |   15s  <- best
epoch   2 | tr_loss  5.274 | val_loss  4.086 | CER 1.0000 | WER 1.0000 | non-blank   0% | lr 1.2e-04 |   13s
epoch   3 | tr_loss  3.808 | val_loss  3.548 | CER 1.0000 | WER 1.0000 | non-blank   0% | lr 1.8e-04 |   13s
epoch   4 | tr_loss  3.711 | val_loss  3.522 | CER 1.0000 | WER 1.0000 | non-blank   0% | lr 2.4e-04 |   13s
epoch   5 | tr_loss  3.712 | val_loss  3.537 | CER 1.0000 | WER 1.0000 | non-blank   0% | lr 3.0e-04 |   13s
epoch   6 | tr_loss  3.697 | val_loss  3.495 | CER 1.0000 | WER 1.0000 | non-blank   0% | lr 3.0e-04 |   14s
epoch   7 | tr_loss  3.682 | val_loss  3.495 | CER 1.0000 | WER 1.0000 | non-blank   0% | lr 3.0e-04 |   14s
epoch   8 | tr_loss  3.666 | val_loss  3.563 | CER 1.0000 | WER 1.0000 | non-blank   0% | lr 3.0e-04 |   15s
epoch   9 | tr_loss  3.653 | val_loss  3.477 | CER 1.0000 | WER 1.0000 | non-blank   0% | lr 3.0e-04 |   14s
epoch  10 

## Step 9 — Evaluate & inspect predictions

In [11]:
val_loss, val_cer, val_wer, refs, preds = evaluate(val_loader)
exact = sum(r == p for r, p in zip(refs, preds)) / max(1, len(refs))
print(f'Final  ->  CER {val_cer:.4f} | WER {val_wer:.4f} | exact-match {exact:.3f}')
print('(TrOCR-base on the same data reached CER ~0.83 / exact ~0.007)')
print()
for i in range(min(PRINT_TEST_EXAMPLES, len(refs))):
    print(f'[{i+1}] GT: {refs[i]}')
    print(f'    PR: {preds[i]}')

Final  ->  CER 0.8941 | WER 0.9968 | exact-match 0.010
(TrOCR-base on the same data reached CER ~0.83 / exact ~0.007)

[1] GT: P.S. Zygmunt ma zawsze na ulicy minę strasznie zafrasowaną.
    PR: c      
[2] GT: chczytaję zdrowia wytaję
    PR: c    
[3] GT: chcól konie bardzo zdrowia, kto zgodzić
    PR: c       
[4] GT: chciskam wytaję za bardzo, bo bg. Pragnę
    PR: c    
[5] GT: Pa, mój Staś kochany - więcej na dobre - poniewczasie przy swojej Reni
    PR: ch         
[6] GT: Poznań 15/XII. 1925 r. godz 2giej noc. Oczekuję powrotu
    PR: d       
[7] GT: w ten sposób dowiadujemy się o
    PR: c  
[8] GT: s' Wes – nie pozdrowić Sniżło Może wyjadę z ten w trzy lat
    PR: m   
[9] GT: Tadnie mogła się labecić. Jakd Ty się nie uyes o gody futhem
    PR: c       
[10] GT: W marcu r. 1877 donosi z
    PR: c    
[11] GT: ..................................................................
    PR: d
[12] GT: niech Was Bóg ma w swojej opiece.
    PR: ch     
[13] GT: chczytaję z który które

## Step 10 — Save the model to Drive

In [12]:
OUTPUT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
torch.save({
    'model_state': model.state_dict(),
    'char_to_idx': char_to_idx,
    'idx_to_char': idx_to_char,
    'num_classes': NUM_CLASSES,
    'img_height': IMG_HEIGHT,
    'max_width': MAX_WIDTH,
    'downsample': DOWNSAMPLE,
    'best_cer': best_cer,
}, OUTPUT_MODEL_DIR / 'crnn_ctc.pt')
print('Saved to', OUTPUT_MODEL_DIR / 'crnn_ctc.pt')

Saved to /content/drive/MyDrive/UMwTI/models/crnn_ctc_sample1000/crnn_ctc.pt


## Step 11 (optional) — Decode with a Polish n-gram language model

CTC greedy decoding has **no** notion of Polish spelling. A word-level KenLM n-gram
trained on your transcriptions (and optionally a bigger Polish corpus) re-scores beam
hypotheses at decode time and typically shaves a few CER points — exactly the
"bolt on an LM at decode time" trick.

This section is **best-effort and optional**: the model already works without it. If
the KenLM build fails on your Colab runtime, just skip these cells — everything above
still runs end to end.

In [15]:
# Build KenLM (needs apt build deps) + pyctcdecode. Wrapped in try/except so a failure
# here never breaks the notebook.
LM_OK = False
try:
    !apt-get -qq install -y build-essential cmake libboost-system-dev libboost-thread-dev libboost-program-options-dev libboost-test-dev libeigen3-dev zlib1g-dev libbz2-dev liblzma-dev > /dev/null
    !pip install -q https://github.com/kpu/kenlm/archive/master.zip pyctcdecode
    import kenlm  # noqa
    LM_OK = True
    print('KenLM + pyctcdecode ready')
except Exception as e:
    print('LM deps unavailable, skip this section:', e)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
KenLM + pyctcdecode ready


In [14]:
# Train a 5-gram word LM on the transcriptions and build a pyctcdecode beam decoder.
if LM_OK:
    import subprocess, os
    corpus_path = '/content/lm_corpus.txt'
    with open(corpus_path, 'w', encoding='utf-8') as f:
        for r in records:
            f.write(r['text'].lower() + '\n')
    # locate the lmplz/build_binary that pip-installed kenlm compiled
    import kenlm, glob
    kdir = os.path.dirname(kenlm.__file__)
    lmplz = next(iter(glob.glob('/usr/local/bin/lmplz') + glob.glob('/usr/bin/lmplz') + glob.glob(kdir + '/**/lmplz', recursive=True)), 'lmplz')
    arpa = '/content/polish_5gram.arpa'
    os.system(f'{lmplz} -o 5 --discount_fallback < {corpus_path} > {arpa}')
    from pyctcdecode import build_ctcdecoder
    # label list: index 0 is blank -> empty string for pyctcdecode
    labels = [''] + [idx_to_char[i] for i in range(1, NUM_CLASSES)]
    decoder = build_ctcdecoder(labels, kenlm_model_path=arpa, alpha=0.5, beta=1.0)
    print('Beam decoder ready')

OSError: Cannot read model '/content/polish_5gram.arpa' (End of file Byte: 0)

In [ ]:
# Compare greedy vs LM beam search on the validation set.
if LM_OK:
    model.eval()
    refs_lm, greedy_p, beam_p = [], [], []
    with torch.no_grad():
        for imgs, targets, in_len, tgt_len, texts in val_loader:
            imgs = imgs.to(device)
            logits = model(imgs)
            T = logits.size(0)
            in_len_c = torch.clamp(in_len, max=T)
            greedy_p += greedy_decode(logits, in_len_c)
            probs = F.log_softmax(logits, dim=2).exp().permute(1, 0, 2).cpu().numpy()  # (B,T,C)
            for b in range(probs.shape[0]):
                L = int(in_len_c[b])
                beam_p.append(decoder.decode(probs[b, :L]))
            refs_lm += list(texts)
    print(f'greedy     CER {cer(refs_lm, greedy_p):.4f} | WER {wer(refs_lm, greedy_p):.4f}')
    print(f'LM beam    CER {cer(refs_lm, beam_p):.4f} | WER {wer(refs_lm, beam_p):.4f}')
    print()
    for i in range(min(10, len(refs_lm))):
        print(f'[{i+1}] GT    : {refs_lm[i]}')
        print(f'    greedy: {greedy_p[i]}')
        print(f'    beam  : {beam_p[i]}')